# BriefSync: Fine-tuning T5-small on SAMSum (dialogue summarization)

Run this in Google Colab with a free GPU: **Runtime > Change runtime type > T4 GPU**.

Upload `samsum-train.csv`, `samsum-validation.csv` (from your `brief-sync` folder) when prompted by the cell below.

This continues your own preprocessing (random sample + lowercase/whitespace cleaning) and adds the missing pieces: tokenization, the actual `Seq2SeqTrainer` fine-tuning loop, real ROUGE evaluation, and saving the model in the `./saved_summary_model` layout the Summarizer-HF style `app.py` expects.

In [1]:
!pip -q install transformers datasets evaluate rouge_score sentencepiece accelerate

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
from google.colab import files
uploaded = files.upload()  # select samsum-train.csv and samsum-validation.csv

ModuleNotFoundError: No module named 'google'

In [ ]:
import re
import numpy as np
import pandas as pd
import evaluate
import torch
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset

print('GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

MODEL_NAME = 't5-small'
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 64

In [ ]:
train_data = pd.read_csv('samsum-train.csv').dropna(subset=['dialogue', 'summary'])
val_data = pd.read_csv('samsum-validation.csv').dropna(subset=['dialogue', 'summary'])

# same seed/sizes as your own notebook
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

def clean_data(text):
    text = re.sub(r'\r\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'<.*?>', '', text)
    return text.strip().lower()

train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)
val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

train_data['input_text'] = 'summarize: ' + train_data['dialogue']
val_data['input_text'] = 'summarize: ' + val_data['dialogue']

train_ds = Dataset.from_pandas(train_data[['input_text', 'summary']])
val_ds = Dataset.from_pandas(val_data[['input_text', 'summary']])
print(len(train_ds), len(val_ds))

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    model_inputs = tokenizer(batch['input_text'], max_length=MAX_INPUT_LEN, truncation=True, padding='max_length')
    labels = tokenizer(text_target=batch['summary'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length')
    labels['input_ids'] = [[(t if t != tokenizer.pad_token_id else -100) for t in seq] for seq in labels['input_ids']]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

args = Seq2SeqTrainingArguments(
    output_dir='./briefsync_checkpoints',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=collator, compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result.metrics)

In [ ]:
eval_metrics = trainer.evaluate()
print('Final ROUGE / eval metrics:', eval_metrics)

model.save_pretrained('./saved_summary_model')
tokenizer.save_pretrained('./saved_summary_model')

import shutil
shutil.make_archive('saved_summary_model', 'zip', 'saved_summary_model')
files.download('saved_summary_model.zip')

In [ ]:
# quick sanity check: summarize one dialogue
sample_dialogue = val_data['dialogue'].iloc[0]
inputs = tokenizer('summarize: ' + sample_dialogue, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
out = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4, early_stopping=True)
print('DIALOGUE:', sample_dialogue[:300])
print('\nREFERENCE :', val_data['summary'].iloc[0])
print('GENERATED :', tokenizer.decode(out[0], skip_special_tokens=True))